In [1]:
# ============================================================
# Cell 1: Import Libraries
# ============================================================

import os
import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf

print("TensorFlow Version :", tf.__version__)
print("OpenCV Version     :", cv2.__version__)

TensorFlow Version : 2.20.0
OpenCV Version     : 4.13.0


In [2]:
# ============================================================
# Cell 2: Configuration
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)

print("Seed :", SEED)
print("Image Size :", IMG_SIZE)

Seed : 42
Image Size : (224, 224)


In [3]:
# ============================================================
# Cell 3: Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# ============================================================
# Cell 4: Dataset Paths
# ============================================================

ROOT_DIR = "/content/drive/MyDrive/Colab Notebooks/chest_xray"

TRAIN_DIR = os.path.join(ROOT_DIR, "train")
VAL_DIR   = os.path.join(ROOT_DIR, "val")
TEST_DIR  = os.path.join(ROOT_DIR, "test")

print("Train :", TRAIN_DIR)
print("Validation :", VAL_DIR)
print("Test :", TEST_DIR)

assert os.path.exists(TRAIN_DIR)
assert os.path.exists(VAL_DIR)
assert os.path.exists(TEST_DIR)

print("✅ Original Dataset Found")

Train : /content/drive/MyDrive/Colab Notebooks/chest_xray/train
Validation : /content/drive/MyDrive/Colab Notebooks/chest_xray/val
Test : /content/drive/MyDrive/Colab Notebooks/chest_xray/test
✅ Original Dataset Found


In [5]:
# ============================================================
# Cell 5: Dataset Summary
# ============================================================

import os

def count_images(folder):
    total = 0
    for root, _, files in os.walk(folder):
        total += len([f for f in files if f.lower().endswith((".jpeg", ".jpg", ".png"))])
    return total

train_count = count_images(TRAIN_DIR)
val_count = count_images(VAL_DIR)
test_count = count_images(TEST_DIR)

print("=" * 50)
print("Dataset Summary")
print("=" * 50)

print(f"Train Images      : {train_count}")
print(f"Validation Images : {val_count}")
print(f"Test Images       : {test_count}")
print(f"Total Images      : {train_count + val_count + test_count}")

Dataset Summary
Train Images      : 5216
Validation Images : 16
Test Images       : 624
Total Images      : 5856


In [6]:
# ============================================================
# Cell 6: Folder Structure
# ============================================================

for folder in [TRAIN_DIR, VAL_DIR, TEST_DIR]:

    print(f"\n📂 {os.path.basename(folder).upper()}")

    for cls in sorted(os.listdir(folder)):
        path = os.path.join(folder, cls)

        if os.path.isdir(path):
            n = len([
                f for f in os.listdir(path)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ])

            print(f"   {cls:<12} : {n}")


📂 TRAIN
   NORMAL       : 1341
   PNEUMONIA    : 3875

📂 VAL
   NORMAL       : 8
   PNEUMONIA    : 8

📂 TEST
   NORMAL       : 234
   PNEUMONIA    : 390


In [7]:
# ============================================================
# Cell 7: Check Corrupted Images
# ============================================================

from PIL import Image

corrupted = []

for folder in [TRAIN_DIR, VAL_DIR, TEST_DIR]:

    for root, _, files in os.walk(folder):

        for file in files:

            if file.lower().endswith((".jpg", ".jpeg", ".png")):

                path = os.path.join(root, file)

                try:
                    img = Image.open(path)
                    img.verify()

                except Exception:
                    corrupted.append(path)

print("Corrupted Images :", len(corrupted))

if len(corrupted) == 0:
    print("✅ All Images are Valid")

Corrupted Images : 0
✅ All Images are Valid


In [8]:
# ============================================================
# Cell 8: Image Size Statistics
# ============================================================

sizes = []

for folder in [TRAIN_DIR, VAL_DIR, TEST_DIR]:

    for root, _, files in os.walk(folder):

        for file in files:

            if file.lower().endswith((".jpg", ".jpeg", ".png")):

                img = Image.open(os.path.join(root, file))

                sizes.append(img.size)

widths = [w for w, h in sizes]
heights = [h for w, h in sizes]

print("Minimum Width :", min(widths))
print("Maximum Width :", max(widths))

print("Minimum Height :", min(heights))
print("Maximum Height :", max(heights))

Minimum Width : 384
Maximum Width : 2916
Minimum Height : 127
Maximum Height : 2713


In [9]:
# ============================================================
# Cell 9: Create Output Directory
# ============================================================

OUTPUT_DIR = "/content/drive/MyDrive/Research paper/Pneumonia_Project/processed_dataset_v2"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output Folder:")
print(OUTPUT_DIR)

Output Folder:
/content/drive/MyDrive/Research paper/Pneumonia_Project/processed_dataset_v2


In [10]:
# ============================================================
# Cell 10: Collect All Images
# ============================================================

from pathlib import Path

dataset = {
    "NORMAL": [],
    "BACTERIA": [],
    "VIRUS": []
}

# ---------- TRAIN ----------
for cls in ["NORMAL", "PNEUMONIA"]:

    folder = Path(TRAIN_DIR) / cls

    for img in folder.glob("*"):

        if cls == "NORMAL":
            dataset["NORMAL"].append(str(img))

        else:
            name = img.name.upper()

            if "BACTERIA" in name:
                dataset["BACTERIA"].append(str(img))
            else:
                dataset["VIRUS"].append(str(img))


# ---------- TEST ----------
for cls in ["NORMAL", "PNEUMONIA"]:

    folder = Path(TEST_DIR) / cls

    for img in folder.glob("*"):

        if cls == "NORMAL":
            dataset["NORMAL"].append(str(img))

        else:
            name = img.name.upper()

            if "BACTERIA" in name:
                dataset["BACTERIA"].append(str(img))
            else:
                dataset["VIRUS"].append(str(img))


# ---------- VALIDATION ----------
for cls in ["NORMAL", "PNEUMONIA"]:

    folder = Path(VAL_DIR) / cls

    for img in folder.glob("*"):

        if cls == "NORMAL":
            dataset["NORMAL"].append(str(img))

        else:
            name = img.name.upper()

            if "BACTERIA" in name:
                dataset["BACTERIA"].append(str(img))
            else:
                dataset["VIRUS"].append(str(img))


print("=" * 50)

for cls in dataset:

    print(f"{cls:10s} : {len(dataset[cls])}")

print("=" * 50)
print("Total Images :", sum(len(v) for v in dataset.values()))

NORMAL     : 1583
BACTERIA   : 2780
VIRUS      : 1493
Total Images : 5856


In [11]:
# ============================================================
# Cell 11: Stratified Train / Validation / Test Split
# ============================================================

from sklearn.model_selection import train_test_split

split_dataset = {}

for cls, images in dataset.items():

    # First split: 70% train, 30% temporary
    train_imgs, temp_imgs = train_test_split(
        images,
        test_size=0.30,
        random_state=SEED,
        shuffle=True
    )

    # Second split: 15% validation, 15% test
    val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=0.50,
        random_state=SEED,
        shuffle=True
    )

    split_dataset[cls] = {
        "train": train_imgs,
        "val": val_imgs,
        "test": test_imgs
    }

print("=" * 60)

for cls in split_dataset:

    print(f"\n{cls}")

    print(f" Train : {len(split_dataset[cls]['train'])}")

    print(f" Val   : {len(split_dataset[cls]['val'])}")

    print(f" Test  : {len(split_dataset[cls]['test'])}")

print("=" * 60)


NORMAL
 Train : 1108
 Val   : 237
 Test  : 238

BACTERIA
 Train : 1946
 Val   : 417
 Test  : 417

VIRUS
 Train : 1045
 Val   : 224
 Test  : 224


In [12]:
# ============================================================
# Cell 12: Verify New Split
# ============================================================

total_train = sum(len(split_dataset[c]["train"]) for c in split_dataset)
total_val = sum(len(split_dataset[c]["val"]) for c in split_dataset)
total_test = sum(len(split_dataset[c]["test"]) for c in split_dataset)

print("=" * 50)
print("New Dataset Split")
print("=" * 50)

print(f"Train      : {total_train}")
print(f"Validation : {total_val}")
print(f"Test       : {total_test}")
print(f"Total      : {total_train + total_val + total_test}")

New Dataset Split
Train      : 4099
Validation : 878
Test       : 879
Total      : 5856


In [13]:
# ============================================================
# Cell 13: Create processed_dataset_v2 Structure
# ============================================================

import os

DATASET_V2 = "/content/drive/MyDrive/Research paper/Pneumonia_Project/processed_dataset_v2"

for split in ["train", "val", "test"]:

    for cls in ["NORMAL", "BACTERIA", "VIRUS"]:

        os.makedirs(
            os.path.join(DATASET_V2, split, cls),
            exist_ok=True
        )

print("✅ Folder Structure Created")

print(DATASET_V2)

✅ Folder Structure Created
/content/drive/MyDrive/Research paper/Pneumonia_Project/processed_dataset_v2


In [14]:
# ============================================================
# Cell 14: Resize Images and Save Dataset
# ============================================================

from PIL import Image
from tqdm import tqdm

IMG_SIZE = (224, 224)

for cls in split_dataset:

    for split in ["train", "val", "test"]:

        image_list = split_dataset[cls][split]

        save_dir = os.path.join(DATASET_V2, split, cls)

        for img_path in tqdm(image_list, desc=f"{split}-{cls}"):

            img = Image.open(img_path)

            img = img.convert("RGB")

            img = img.resize(IMG_SIZE)

            filename = os.path.basename(img_path)

            img.save(
                os.path.join(save_dir, filename),
                quality=95
            )

print("\n✅ processed_dataset_v2 Created Successfully")

test-VIRUS: 100%|██████████| 224/224 [00:15<00:00, 14.68it/s]


✅ processed_dataset_v2 Created Successfully


In [15]:
# ============================================================
# Cell 15: Verify processed_dataset_v2
# ============================================================

def count_images(folder):

    total = 0

    for root, _, files in os.walk(folder):

        total += len([
            f for f in files
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    return total

print("=" * 50)

for split in ["train", "val", "test"]:

    folder = os.path.join(DATASET_V2, split)

    print(f"{split.upper()} : {count_images(folder)}")

print("=" * 50)

TRAIN : 4099
VAL : 878
TEST : 879
